# MATTERSIM metallic benchmark

Eight task sections use the same numerical workflows. This is consolidated future-rerun code, not proof of exact reproduction of historical predictions. Enable only selected tasks. See README.md for schema and protocol details.

In [ ]:
from pathlib import Path
import os
import sys
HERE = Path.cwd().resolve()
MODULE_DIR = next(p for p in [HERE, HERE/'notebooks', HERE.parent/'notebooks']
                  if (p/'benchmark_workflows.py').is_file())
sys.path.insert(0,str(MODULE_DIR))
import benchmark_workflows as workflows
REPOSITORY = workflows.repo_root()

MODEL = 'mattersim'
CHECKPOINT = Path(os.environ.get('MATTERSIM_CHECKPOINT',str(REPOSITORY/'model'/'default_5M.pth')))
DEVICE = os.environ.get('BENCHMARK_DEVICE','cpu')
MODEL_OPTIONS = {}
# DPA requires a verified checkpoint-specific type map, not an assumed element order.
OUTPUT_ROOT = REPOSITORY/'results'/'notebook-reruns'/MODEL
LOAD_MODEL = False
calculator = None
MODEL_IDENTITY = None
if LOAD_MODEL:
    calculator = workflows.make_calculator(MODEL,CHECKPOINT,DEVICE,MODEL_OPTIONS)
    MODEL_IDENTITY = dict(model=MODEL,checkpoint_sha256=workflows.sha(CHECKPOINT),
                          device=DEVICE,options=MODEL_OPTIONS)
    workflows.dump(OUTPUT_ROOT/'checkpoint.json',dict(**MODEL_IDENTITY,versions=workflows.versions()))


## Static energy and forces

Single-point total energy, energy per atom and unconstrained forces.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_STATIC = False
STATIC_MANIFEST = REPOSITORY/'inputs'/'static.csv'
if RUN_STATIC:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    static_results = workflows.run_task(
        'static', STATIC_MANIFEST, OUTPUT_ROOT/'static', calculator,
        parameters={}, model_identity=MODEL_IDENTITY)


## Bulk geometry optimization

Relax positions and cell. No implicit crystal-symmetry projection.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_BULK = False
BULK_MANIFEST = REPOSITORY/'inputs'/'bulk.csv'
if RUN_BULK:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    bulk_results = workflows.run_task(
        'bulk', BULK_MANIFEST, OUTPUT_ROOT/'bulk', calculator,
        parameters={'fmax': 0.01, 'max_steps': 500}, model_identity=MODEL_IDENTITY)


## Elastic stiffness and VRH moduli

Six engineering-strain components, centered +/-1% differences, full6x6 stiffness and VRH. Strained internal coordinates are clamped.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_ELASTIC = False
ELASTIC_MANIFEST = REPOSITORY/'inputs'/'elastic.csv'
if RUN_ELASTIC:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    elastic_results = workflows.run_task(
        'elastic', ELASTIC_MANIFEST, OUTPUT_ROOT/'elastic', calculator,
        parameters={'fmax': 0.01, 'max_steps': 500, 'strain': 0.01}, model_identity=MODEL_IDENTITY)


## Vacancy formation energy

Explicit matched cells, removed species and chemical potential. Fixed-cell relaxations.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_VACANCY = False
VACANCY_MANIFEST = REPOSITORY/'inputs'/'vacancy.csv'
if RUN_VACANCY:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    vacancy_results = workflows.run_task(
        'vacancy', VACANCY_MANIFEST, OUTPUT_ROOT/'vacancy', calculator,
        parameters={'fmax': 0.027, 'max_steps': 500}, model_identity=MODEL_IDENTITY)


## Stacking-fault energy

Explicit matched paths, fault_type and fault_planes. Fixed cell with input atomic constraints. Use final I1 with F F T, never historical I1-old.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_SFE = False
SFE_MANIFEST = REPOSITORY/'inputs'/'sfe.csv'
if RUN_SFE:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    sfe_results = workflows.run_task(
        'sfe', SFE_MANIFEST, OUTPUT_ROOT/'sfe', calculator,
        parameters={'fmax': 0.02, 'max_steps': 500}, model_identity=MODEL_IDENTITY)


## Unrelaxed surface energy

Matched slab and bulk, explicit exposed-surface count and stoichiometry validation.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_SURFACE = False
SURFACE_MANIFEST = REPOSITORY/'inputs'/'surface.csv'
if RUN_SURFACE:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    surface_results = workflows.run_task(
        'surface', SURFACE_MANIFEST, OUTPUT_ROOT/'surface', calculator,
        parameters={}, model_identity=MODEL_IDENTITY)


## Relaxation efficiency

Fixed-cell 100-step convergence and GPU-synchronized wall-clock measurements.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_EFFICIENCY = False
EFFICIENCY_MANIFEST = REPOSITORY/'inputs'/'efficiency.csv'
if RUN_EFFICIENCY:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    efficiency_results = workflows.run_task(
        'efficiency', EFFICIENCY_MANIFEST, OUTPUT_ROOT/'efficiency', calculator,
        parameters={'fmax': 0.015, 'max_steps': 100}, model_identity=MODEL_IDENTITY)


## Ideal tensile stress-strain

Simulation z-axis loading (axis=2), matching the original codes. Relax transverse x/y normal strains and analyze sigma_zz. Input z must align with the intended cubic <100> family direction. Cleaned rerun stress convention is ASE tensile-positive; no automatic sign flip.
Prepare task CSVs following config/manifest_examples. All RUN flags default to False.

In [ ]:
RUN_TENSILE = False
TENSILE_MANIFEST = REPOSITORY/'inputs'/'tensile.csv'
if RUN_TENSILE:
    if calculator is None:
        raise RuntimeError('Set LOAD_MODEL=True and execute the configuration cell.')
    tensile_results = workflows.run_task(
        'tensile', TENSILE_MANIFEST, OUTPUT_ROOT/'tensile', calculator,
        parameters={'axis': 2, 'strain_step': 0.01, 'max_strain': 0.5, 'fmax': 0.01, 'max_steps': 200}, model_identity=MODEL_IDENTITY)
